# First timelapse — any index, any AOI

Build a cloud-masked, monthly-median satellite animation over a forest AOI in a **few lines**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/prerelease/single-user-intro/notebooks/01_first_timelapse.ipynb)

Steps: **install → authenticate once → load an AOI → `animate(...)`**. Makes live Earth Engine calls
under **your** project — [00_setup](00_setup.ipynb) gets you one if you don't have it yet.

## 1 · Install (Colab only)

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "prerelease/single-user-intro"   # flipped to "main" when this PR merges
URL = f"gee_animation[shapefile] @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package (first run)
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code
# pip resolves by version (0.2.0), which doesn't change per commit on a branch --
# --force-reinstall pulls the latest commit. After a code update: Runtime ->
# Restart session, then Run all, so the new module is re-imported.

> **Updated the code on the branch?** After the install cell above, do **Runtime → Restart session**, then **Run all** — otherwise Python keeps the previously-imported (old) module in memory.

## 2 · Authenticate Earth Engine (once)
A separate step — run it a single time. In Colab it opens an `ee.Authenticate()` prompt.
No project of your own yet? [00_setup](00_setup.ipynb#2-·-create-your-own-earth-engine-project)
walks you through creating one (five minutes, free).

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"

from gee_animation import auth
auth.init(PROJECT)

## 3 · Load an area of interest
Here we download the example **WNE forest** AOI from the repo and load it from disk. `animate()` reads the GeoJSON straight from the path.

In [ ]:
import urllib.request

AOI_URL = "https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/prerelease/single-user-intro/docs/aoi/wne/wne.geojson"
urllib.request.urlretrieve(AOI_URL, "wne.geojson")   # download to the working dir
REGION = "wne.geojson"                                # load from disk

# --- Use your OWN AOI instead of the line above: ---
#   • upload a file in Colab:
#       from google.colab import files; files.upload()
#       REGION = "my_aoi.geojson"
#   • or point at any local GeoJSON path:  REGION = "/content/my_aoi.geojson"
#   • or a bounding box (no file needed):  REGION = [13.8, 52.9, 14.0, 53.05]  # [W, S, E, N]

## 4 · Choose products & dates

In [ ]:
# (sensor, index) pairs. Valid: sentinel2/landsat/modis + ndvi/evi/ndwi/ndmi/rgb/cir;
# landsat + lst/lst_smw/lst_sharp; modis_lst + lst_modis.
PRODUCTS = [("landsat", "lst"), ("sentinel2", "ndvi")]
START, END = "2023-01-01", "2024-01-01"
BUFFER_M, PRESET = 1000, "1080p"   # PRESET upscales native imagery so LST/MODIS are not tiny
FPS = 1                            # frames/sec — lower is slower (0.5 = 2 s per frame)

## 5 · Generate & show
`animate()` does AOI → build → monthly median → render, and plays inline.

In [ ]:
from gee_animation import animate
from IPython.display import display

anims = []
for sensor, index in PRODUCTS:
    anim = animate(REGION, sensor=sensor, index=index, start=START, end=END,
                   buffer_m=BUFFER_M, preset=PRESET, fps=FPS, project=PROJECT)
    display(anim)          # inline video
    anims.append(anim)

## 6 · Everything is on the `anim` object
`display(anim)` already showed the video **and** the inside-vs-outside-AOI chart. You can also pull the pieces back out later:

In [ ]:
a = anims[0]
print(a.status)
print('files:', a.mp4, a.chart, a.metadata_db)
a.series            # [(month, inside_mean, outside_mean), ...]
a.metadata()        # [{month, n_scenes, aoi_cloud_fraction, aoi_mean}, ...] from the DB

## 7 · Download (Colab)

In [ ]:
if IS_COLAB:
    from google.colab import files
    for a in anims:
        if a.mp4:
            files.download(a.mp4)

## Make it yours

1. **Your AOI.** Replace the download in §3 with your own file — in Colab:
   `from google.colab import files; files.upload()`, then `REGION = "my_aoi.geojson"`.
   A zipped shapefile works too (`REGION = "my_aoi.zip"`).
2. **Disturbance instead of greenness.** Add `("sentinel2", "nbr")` to `PRODUCTS` —
   the normalized burn ratio, the damage-and-recovery view of the same months.
   (`landsat`/`lst` is already in the list — that one is temperature, and
   notebook 03 shows what its 100 m resolution really means.)
3. **The numbers behind the picture.** Pass `geotiffs=True` to `animate(...)` and
   every monthly frame is also written as a georeferenced GeoTIFF under
   `out/geotiffs/<name>/` — open one in QGIS and inspect the values the colours encode.

*The seam behind it:* the `Animation` object you got back carries everything —
`anim.frames` (the PNGs), `anim.series` (the inside/outside numbers),
`anim.metadata()` (scene counts and cloud fractions per month). Nothing shown
inline is lost; it is all files and lists you can process further.